# LASSO / Elastic Net Feature Selection (LOOCV)

Another simple alternative to `Feature_Weighting.ipynb`. Runs both pure LASSO
(L1-penalized logistic regression, `l1_ratio=1.0`) and Elastic Net
(`l1_ratio` between 0 and 1) side by side, leave-one-out cross-validated (8
folds) per sex — same pattern as `RandomForest_Feature_Importance.ipynb`.

LASSO's L1 penalty zeroes out most gene coefficients, but it arbitrarily
picks just one gene from a group of correlated genes and drops the rest.
Elastic Net tends to keep or drop correlated genes together instead — worth
comparing directly since real correlated gene pairs turned up among the top
attribute-weighting hits (e.g. Dbp-Ky r=0.95).

Both tables also get a `HedgesG` column — a small-sample-corrected effect
size computed directly from expression values, independent of either model,
so it's directly comparable between the two.

**Real data guard:** `USE_REAL_DATA` must stay `False` in this AI session. Only set `True` for your own local, offline run (see `CLAUDE.md`).

**Limitations:**
- How many genes get picked depends a lot on one setting, `C` (regularization strength) — the sweep steps below find a sane value for each method instead of guessing. LASSO and Elastic Net need separate sweeps — the same `C` means something different at a different `l1_ratio`.
- Only 8 samples, thousands of genes — treat this as a cross-check, not a replacement for the main pipeline.
- Male and female are kept separate.

In [ ]:
USE_REAL_DATA = True # keep False here. Only set True in your own local run, never inside an AI tool session, per CLAUDE.md.

import time
import pandas as pd

from lasso_selection import run_loocv_lasso, sweep_C, hedges_g

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}
DATA_DIR = "restructured"
TOP_N = 10
ELASTIC_NET_L1_RATIO = 0.5  # 1.0 = pure LASSO, 0.0 = pure Ridge, in between = Elastic Net

## Step 1. Load data and check shapes

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    data[sex] = {"X": df[gene_cols], "y": df["Label"]}
    print(f"{sex}: OK, {df.shape[0]} samples x {len(gene_cols)} genes, 4/4 Group split, Label matches Group")

## Step 2. Sweep C for LASSO (l1_ratio=1.0)

`C` controls how many genes survive. Too small and the model selects 0 genes (gives up and predicts the majority class); too large and it selects hundreds. Look for a `C` with a small `mean_genes_per_fold` (aim for a couple dozen) and good accuracy, then set `CHOSEN_C_LASSO` below.

In [ ]:
C_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.7, 1.0]

for sex in SEXES:
    print(f"--- {sex} (LASSO, l1_ratio=1.0) ---")
    print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID, l1_ratio=1.0))
    print()

## Step 2.5. Sweep C for Elastic Net (l1_ratio=0.5)

Same idea, but a different `C` — Elastic Net's mix of L1/L2 changes how many genes
survive at a given `C`, so LASSO's sweep result above doesn't transfer. Set
`CHOSEN_C_ELASTIC_NET` below from this sweep, not from Step 2's.

In [ ]:
for sex in SEXES:
    print(f"--- {sex} (Elastic Net, l1_ratio={ELASTIC_NET_L1_RATIO}) ---")
    print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID, l1_ratio=ELASTIC_NET_L1_RATIO))
    print()

## Step 3. Sanity check on toy data, using the chosen C for each method

Set `CHOSEN_C_LASSO` and `CHOSEN_C_ELASTIC_NET` from Steps 2 and 2.5 (0.3/0.3 worked on synthetic data — pick your own from the sweeps above). One signal gene, four noise genes — the signal gene should be the only one selected, for both methods.

In [ ]:
import numpy as np

CHOSEN_C_LASSO = 0.7  # set this from the Step 2 sweep on your data, not left as the synthetic default
CHOSEN_C_ELASTIC_NET = 0.7  # set this from the Step 2.5 sweep on your data, not left as the synthetic default

toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

rng = np.random.default_rng(42)
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})

toy_summary, toy_accuracy, _ = run_loocv_lasso(toy_X, toy_y, C=CHOSEN_C_LASSO, l1_ratio=1.0)
print(toy_summary)
print(f"toy LOOCV accuracy (LASSO): {toy_accuracy:.0%}")

assert toy_summary.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
print("PASS — signal gene ranked above all noise genes (LASSO)")

In [ ]:
toy_summary_en, toy_accuracy_en, _ = run_loocv_lasso(toy_X, toy_y, C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO)
print(toy_summary_en)
print(f"toy LOOCV accuracy (Elastic Net): {toy_accuracy_en:.0%}")

assert toy_summary_en.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
print("PASS — signal gene ranked above all noise genes (Elastic Net)")

## Step 4. Run LOOCV LASSO on both sexes

Each fold fits LASSO on 7 samples and predicts the 1 held out. `SelectionFreq` is how often a gene survived with a nonzero coefficient across the 8 folds — genes at 1.0 were picked every time.

In [ ]:
summary_results = {}
accuracy_results = {}
fold_detail_results = {}

for sex in SEXES:
    t0 = time.time()
    summary, accuracy, fold_detail = run_loocv_lasso(data[sex]["X"], data[sex]["y"], C=CHOSEN_C_LASSO, l1_ratio=1.0)
    elapsed = time.time() - t0

    summary_results[sex] = summary
    accuracy_results[sex] = accuracy
    fold_detail_results[sex] = fold_detail

    n_ever_selected = (summary["SelectionFreq"] > 0).sum()
    print(f"--- {sex} (LASSO) ---  ({elapsed:.1f}s, C={CHOSEN_C_LASSO})")
    print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
    print(f"mean genes selected per fold: {fold_detail['n_genes_selected'].mean():.1f}")
    print(f"genes ever selected across folds: {n_ever_selected}")
    print(f"Top {TOP_N} genes by SelectionFreq:")
    print(summary.head(TOP_N))
    print()

## Step 4.5. Run LOOCV Elastic Net on both sexes

Same idea as Step 4, `l1_ratio=ELASTIC_NET_L1_RATIO` instead of pure LASSO.

In [ ]:
summary_results_en = {}
accuracy_results_en = {}
fold_detail_results_en = {}

for sex in SEXES:
    t0 = time.time()
    summary, accuracy, fold_detail = run_loocv_lasso(
        data[sex]["X"], data[sex]["y"], C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO
    )
    elapsed = time.time() - t0

    summary_results_en[sex] = summary
    accuracy_results_en[sex] = accuracy
    fold_detail_results_en[sex] = fold_detail

    n_ever_selected = (summary["SelectionFreq"] > 0).sum()
    print(f"--- {sex} (Elastic Net) ---  ({elapsed:.1f}s, C={CHOSEN_C_ELASTIC_NET}, l1_ratio={ELASTIC_NET_L1_RATIO})")
    print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
    print(f"mean genes selected per fold: {fold_detail['n_genes_selected'].mean():.1f}")
    print(f"genes ever selected across folds: {n_ever_selected}")
    print(f"Top {TOP_N} genes by SelectionFreq:")
    print(summary.head(TOP_N))
    print()

## Step 4.6. Attach Hedges' g effect size to both tables

Computed once per sex directly from expression values (independent of either
model), then joined onto both the LASSO and Elastic Net summary tables — so
the same gene's effect size is directly comparable between the two methods.

In [ ]:
effect_sizes = {}

for sex in SEXES:
    g = hedges_g(data[sex]["X"], data[sex]["y"])
    effect_sizes[sex] = g

    summary_results[sex] = summary_results[sex].join(g.rename("HedgesG"))
    summary_results_en[sex] = summary_results_en[sex].join(g.rename("HedgesG"))

    print(f"--- {sex}: top {TOP_N} by |HedgesG| ---")
    print(g.reindex(g.abs().sort_values(ascending=False).index).head(TOP_N).rename("HedgesG"))
    print()

## Step 5. Save results

Saves the full selection-frequency table (with `HedgesG`), per-fold predictions, and a shortlist (genes selected in every fold) for each sex — separately for LASSO and Elastic Net.

In [ ]:
def save_method_results(method_label, summaries, fold_details):
    for sex in SEXES:
        summary = summaries[sex]

        summary_path = f"{DATA_DIR}/{sex}_{method_label}_selection.csv"
        shortlist_path = f"{DATA_DIR}/{sex}_{method_label}_shortlist_genes.txt"
        fold_detail_path = f"{DATA_DIR}/{sex}_{method_label}_loocv_folds.csv"

        summary.to_csv(summary_path, index_label="Gene")
        fold_details[sex].to_csv(fold_detail_path, index=False)

        shortlist = summary.index[summary["SelectionFreq"] == 1.0].tolist()
        with open(shortlist_path, "w") as f:
            f.write("\n".join(shortlist))

        print(f"{sex} ({method_label}): saved {len(summary)}-gene selection table -> {summary_path}")
        print(f"{sex} ({method_label}): saved {len(shortlist)} genes selected in every fold -> {shortlist_path}")
        print(f"{sex} ({method_label}): saved {len(fold_details[sex])}-fold LOOCV detail -> {fold_detail_path}")

save_method_results("lasso", summary_results, fold_detail_results)
print()
save_method_results("elastic_net", summary_results_en, fold_detail_results_en)

## Step 6. Compare with attribute weighting results

Checks overlap between LASSO/Elastic Net's stable genes (`SelectionFreq == 1.0`)
and attribute weighting's stability-ranked 30-gene shortlist
(`Feature_Weighting.ipynb`, Step 6) — this is the actual point of running two
independent methods on the same data: genes both agree on are your strongest
candidates.

In [ ]:
for sex in SEXES:
    aw_path = f"{DATA_DIR}/{sex}_signature_genes_draft_shortlist.txt"
    try:
        with open(aw_path) as f:
            aw_shortlist = set(line.strip() for line in f if line.strip())
    except FileNotFoundError:
        print(f"{sex}: {aw_path} not found — run Feature_Weighting.ipynb's Step 6 first")
        continue

    lasso_shortlist = set(summary_results[sex].index[summary_results[sex]["SelectionFreq"] == 1.0])
    en_shortlist = set(summary_results_en[sex].index[summary_results_en[sex]["SelectionFreq"] == 1.0])

    print(f"--- {sex} ---")
    print(f"attribute weighting shortlist: {len(aw_shortlist)} genes")
    print(f"LASSO shortlist:               {len(lasso_shortlist)} genes")
    print(f"Elastic Net shortlist:         {len(en_shortlist)} genes")
    print()
    print(f"LASSO ∩ attribute weighting:       {sorted(lasso_shortlist & aw_shortlist)}")
    print(f"Elastic Net ∩ attribute weighting: {sorted(en_shortlist & aw_shortlist)}")
    print(f"LASSO ∩ Elastic Net:               {sorted(lasso_shortlist & en_shortlist)}")
    print()


## Step 7. Permutation-null test for LOOCV accuracy

At n=8 with a balanced 4/4 split, there are only C(8,4)=70 possible balanced
label assignments — few enough to test all of them exactly (not an
approximation). Reruns the full LOOCV pipeline once per permutation, at the
same `C`/`l1_ratio` already chosen from the real data. `p` = fraction of all
70 permutations (including the true one) that reach accuracy at least as
high as the real labels.

**Expected runtime**: ~1-2 min for LASSO per sex, ~5-8 min for Elastic Net per
sex (Elastic Net's `saga` solver is slower per fit).

**Important**: with thousands of genes and only 7 training samples per fold,
even pure noise can hit high LOOCV accuracy by chance — this test might not
come back favorable, and that's a real result to report honestly, not a bug.

In [ ]:
from lasso_selection import permutation_test_accuracy

permutation_results = {}

for sex in SEXES:
    t0 = time.time()
    obs, null_dist, p = permutation_test_accuracy(data[sex]["X"], data[sex]["y"], C=CHOSEN_C_LASSO, l1_ratio=1.0)
    elapsed = time.time() - t0
    permutation_results[("lasso", sex)] = (obs, null_dist, p)
    print(f"--- {sex} (LASSO) ---  ({elapsed:.1f}s, {len(null_dist)} permutations)")
    print(f"observed accuracy: {obs:.0%}   null mean: {null_dist.mean():.0%}   null max: {null_dist.max():.0%}   p={p:.3f}")
    print()

In [ ]:
for sex in SEXES:
    t0 = time.time()
    obs, null_dist, p = permutation_test_accuracy(
        data[sex]["X"], data[sex]["y"], C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO
    )
    elapsed = time.time() - t0
    permutation_results[("elastic_net", sex)] = (obs, null_dist, p)
    print(f"--- {sex} (Elastic Net) ---  ({elapsed:.1f}s, {len(null_dist)} permutations)")
    print(f"observed accuracy: {obs:.0%}   null mean: {null_dist.mean():.0%}   null max: {null_dist.max():.0%}   p={p:.3f}")
    print()

In [ ]:
rows = []
for (method, sex), (obs, null_dist, p) in permutation_results.items():
    rows.append({
        "method": method,
        "sex": sex,
        "observed_accuracy": obs,
        "null_mean_accuracy": null_dist.mean(),
        "null_max_accuracy": null_dist.max(),
        "n_permutations": len(null_dist),
        "p_value": p,
    })

permutation_summary = pd.DataFrame(rows)
permutation_summary_path = f"{DATA_DIR}/permutation_test_accuracy_summary.csv"
permutation_summary.to_csv(permutation_summary_path, index=False)
print(permutation_summary)
print(f"\nsaved -> {permutation_summary_path}")